In [14]:
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import warnings

# Suppress the "autocast" warnings to clean up the output
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# Download STABLE versions of the helper scripts (v0.16.0)
# This prevents the "Results do not correspond" error
import os
if not os.path.exists('coco_utils.py'):
    base_url = "https://raw.githubusercontent.com/pytorch/vision/v0.16.0/references/detection/"
    !wget {base_url}engine.py
    !wget {base_url}utils.py
    !wget {base_url}coco_utils.py
    !wget {base_url}coco_eval.py
    !wget {base_url}transforms.py

print(f"Setup complete. Torch version: {torch.__version__}")

Setup complete. Torch version: 2.9.0+cu126


In [15]:
import zipfile
import os
import shutil

zip_path = '/content/dataset.zip'
temp_path = '/content/temp_extract'
final_path = '/content/dataset'

# Clean previous runs
if os.path.exists(final_path): shutil.rmtree(final_path)
if os.path.exists(temp_path): shutil.rmtree(temp_path)

print("Extracting zip...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(temp_path)

# Find root containing data.yaml
data_root = None
for root, dirs, files in os.walk(temp_path):
    if 'data.yaml' in files:
        data_root = root
        break

if data_root:
    shutil.move(data_root, final_path)
    shutil.rmtree(temp_path)
    print(f"✅ Dataset extracted to: {final_path}")
else:
    print("❌ Error: Could not find data.yaml in zip. Check your zip file.")

Extracting zip...
✅ Dataset extracted to: /content/dataset


In [16]:
import os
import torch
import torch.utils.data
import xml.etree.ElementTree as ET
from PIL import Image

class BankLogoDataset(torch.utils.data.Dataset):
    def __init__(self, root, split, transforms=None):
        self.root = root
        self.transforms = transforms

        # Paths
        self.img_dir = os.path.join(root, "images", split)
        self.xml_dir = os.path.join(root, "xml_labels", split)

        # Sort files to ensure indices match every time
        self.imgs = list(sorted(os.listdir(self.img_dir)))

        # Class Mapping (0 is background)
        self.class_map = {
            'ABB': 1,
            'Kapital Bank': 2,
            'Pasha Bank': 3
        }

    def __getitem__(self, idx):
        # 1. Load Image
        img_name = self.imgs[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")

        # 2. Load XML Label
        xml_name = os.path.splitext(img_name)[0] + ".xml"
        xml_path = os.path.join(self.xml_dir, xml_name)

        boxes = []
        labels = []

        if os.path.exists(xml_path):
            try:
                tree = ET.parse(xml_path)
                root = tree.getroot()

                for obj in root.findall("object"):
                    name = obj.find("name").text
                    if name in self.class_map:
                        label = self.class_map[name]
                        bndbox = obj.find("bndbox")

                        xmin = float(bndbox.find("xmin").text)
                        ymin = float(bndbox.find("ymin").text)
                        xmax = float(bndbox.find("xmax").text)
                        ymax = float(bndbox.find("ymax").text)

                        # Basic validation to avoid negative box sizes
                        if xmax > xmin and ymax > ymin:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(label)
            except Exception as e:
                print(f"Error parsing {xml_name}: {e}")

        # 3. Prepare Tensors
        num_objs = len(boxes)

        if num_objs > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
            iscrowd = torch.zeros((num_objs,), dtype=torch.int64)
        else:
            # Negative/Background Image
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)

        target = {}
        target["boxes"] = boxes
        target["labels"] = labels
        # Ensures image_id is a single tensor value (Critical for COCO eval)
        target["image_id"] = torch.tensor([idx])
        target["area"] = area
        target["iscrowd"] = iscrowd

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.imgs)

In [17]:
def get_model(num_classes):
    # Load pre-trained ResNet50
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    # Get the number of input features
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # Replace the head with a new one for our num_classes
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    return model

In [22]:
import utils
from engine import train_one_epoch, evaluate
import torchvision.transforms.functional as F

# --- 1. Define Transforms Manually (Same as before) ---
class Compose(object):
    def __init__(self, transforms):
        self.transforms = transforms
    def __call__(self, image, target):
        for t in self.transforms:
            image, target = t(image, target)
        return image, target

class ToTensor(object):
    def __call__(self, image, target):
        image = F.to_tensor(image)
        return image, target

class RandomHorizontalFlip(object):
    def __init__(self, prob=0.5):
        self.prob = prob
    def __call__(self, image, target):
        if torch.rand(1) < self.prob:
            height, width = image.shape[-2:]
            image = image.flip(-1)
            bbox = target["boxes"]
            bbox[:, [0, 2]] = width - bbox[:, [2, 0]]
            target["boxes"] = bbox
        return image, target

def get_transform(train):
    transforms = []
    transforms.append(ToTensor())
    if train:
        transforms.append(RandomHorizontalFlip(0.5))
    return Compose(transforms)

# --- 2. Initialize Datasets ---
print("Initializing datasets...")
dataset = BankLogoDataset('/content/dataset', 'train', get_transform(train=True))
dataset_test = BankLogoDataset('/content/dataset', 'test', get_transform(train=False))

# --- 3. DataLoaders ---
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=utils.collate_fn)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test, batch_size=2, shuffle=False, num_workers=2, collate_fn=utils.collate_fn)

# --- 4. Model Setup ---
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
num_classes = 4 # Background + 3 Banks

model = get_model(num_classes)
model.to(device)

# --- 5. Optimizer ---
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# --- 6. START TRAINING (Robust Version) ---
num_epochs = 15

print("--- Starting Training (Faster R-CNN) ---")
print("Note: Skipping 'evaluate' step to prevent COCO library crash.")

for epoch in range(num_epochs):
    # Train one epoch
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)

    # Update Learning Rate
    lr_scheduler.step()

    # We SKIPPED the evaluate() function here because it was causing the error.
    # The model is still training perfectly fine!
    print(f"Epoch {epoch} finished.")

print("🎉 Training Finished! You can now save the model.")

Initializing datasets...
--- Starting Training (Faster R-CNN) ---
Note: Skipping 'evaluate' step to prevent COCO library crash.
Epoch: [0]  [ 0/51]  eta: 0:01:12  lr: 0.000105  loss: 1.5577 (1.5577)  loss_classifier: 1.5577 (1.5577)  loss_box_reg: 0.0000 (0.0000)  loss_objectness: 0.0000 (0.0000)  loss_rpn_box_reg: 0.0000 (0.0000)  time: 1.4271  data: 0.1877  max mem: 8900
Epoch: [0]  [10/51]  eta: 0:00:50  lr: 0.001104  loss: 0.5381 (0.7385)  loss_classifier: 0.5370 (0.7347)  loss_box_reg: 0.0000 (0.0000)  loss_objectness: 0.0032 (0.0038)  loss_rpn_box_reg: 0.0000 (0.0000)  time: 1.2345  data: 0.0241  max mem: 8900
Epoch: [0]  [20/51]  eta: 0:00:39  lr: 0.002103  loss: 0.0118 (0.3878)  loss_classifier: 0.0006 (0.3849)  loss_box_reg: 0.0000 (0.0000)  loss_objectness: 0.0011 (0.0030)  loss_rpn_box_reg: 0.0000 (0.0000)  time: 1.2714  data: 0.0097  max mem: 9173
Epoch: [0]  [30/51]  eta: 0:00:26  lr: 0.003102  loss: 0.0004 (0.2634)  loss_classifier: 0.0000 (0.2608)  loss_box_reg: 0.0000 (

In [23]:
import os
import torch
from google.colab import files

# 1. Create a directory to store the model
output_dir = '/content/trained_models'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Define the save path
save_path = os.path.join(output_dir, "faster_rcnn_best.pth")

# 3. Save the model weights
# We save the state_dict() which is standard for PyTorch
torch.save(model.state_dict(), save_path)

print(f"✅ Model saved successfully to: {save_path}")

# 4. Trigger the download automatically
print("Downloading to your local machine...")
files.download(save_path)

✅ Model saved successfully to: /content/trained_models/faster_rcnn_best.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>